<a href="https://colab.research.google.com/github/orhanaydinn/oax_1B/blob/main/oax_1B_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SECTION 0 — CONNECT GOOGLE DRIVE**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Path to your main project folder inside Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/oax_1B"

import os

if not os.path.exists(PROJECT_ROOT):
    os.makedirs(PROJECT_ROOT)
    print(f"Created main project folder: {PROJECT_ROOT}")
else:
    print(f"Project folder already exists: {PROJECT_ROOT}")


Mounted at /content/drive
Created main project folder: /content/drive/MyDrive/oax_1B


# **SECTION 1 — PROJECT INITIALIZATION**

In [2]:
import os

folders = [
    "tokenizer",
    "pretrain",
    "pretrain/checkpoints",
    "pretrain/logs",
    "sft",
    "sft/data",
    "sft/checkpoints",
    "sft/logs",
    "config",
    "datasets_cache"
]

for folder in folders:
    path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(path, exist_ok=True)
    print(f"✓ Created folder: {path}")

print("\nAll project folders are ready!")

✓ Created folder: /content/drive/MyDrive/oax_1B/tokenizer
✓ Created folder: /content/drive/MyDrive/oax_1B/pretrain
✓ Created folder: /content/drive/MyDrive/oax_1B/pretrain/checkpoints
✓ Created folder: /content/drive/MyDrive/oax_1B/pretrain/logs
✓ Created folder: /content/drive/MyDrive/oax_1B/sft
✓ Created folder: /content/drive/MyDrive/oax_1B/sft/data
✓ Created folder: /content/drive/MyDrive/oax_1B/sft/checkpoints
✓ Created folder: /content/drive/MyDrive/oax_1B/sft/logs
✓ Created folder: /content/drive/MyDrive/oax_1B/config
✓ Created folder: /content/drive/MyDrive/oax_1B/datasets_cache

All project folders are ready!


# **SECTION 2 — DATASET MIX PLAN**

In [31]:
import json
import os

# Dataset karışım oranları (English Only)
dataset_mix = {
    "slimpajama_en": 0.60,      # High-quality web text
    "wikipedia_en": 0.10,       # Clean encyclopedic knowledge
    "c4_owt_en": 0.10,          # Diverse web text
    "stackexchange_en": 0.10,   # Human Q&A logic
    "code_en": 0.10             # Stabilization + reasoning patterns
}

# Dosyanın kaydedileceği yol
mix_path = os.path.join(PROJECT_ROOT, "config", "dataset_mix.json")

# JSON olarak kaydet
with open(mix_path, "w") as f:
    json.dump(dataset_mix, f, indent=4)

print("Dataset mix saved to:", mix_path)
print("\nDataset Mix:")
print(json.dumps(dataset_mix, indent=4))


Dataset mix saved to: /content/drive/MyDrive/oax_1B/config/dataset_mix.json

Dataset Mix:
{
    "slimpajama_en": 0.6,
    "wikipedia_en": 0.1,
    "c4_owt_en": 0.1,
    "stackexchange_en": 0.1,
    "code_en": 0.1
}


# **SECTION 3 — STREAMING DATASET SETUP**

In [46]:
import json
import os

PROJECT_ROOT = "/content/drive/MyDrive/oax_1B"
os.makedirs(os.path.join(PROJECT_ROOT, "config"), exist_ok=True)

dataset_sources = {
    "slimpajama_en": {
        "hf_path": "cerebras/SlimPajama-627B",
        "split": "train",
        "language_filter": "en"
    },
    "wikipedia_en": {
        "hf_path": "wikimedia/wikipedia",
        "split": "20231101.en",
        "language_filter": "en"
    },
    "c4_owt_en": {
        "hf_path": "allenai/c4",
        "split": "en",
        "language_filter": "en"
    },
    "stackexchange_en": {
        "hf_path": "bigscience-data/roots_code_stackexchange",
        "split": "train",   # <-- DOĞRU SPLIT
        "language_filter": "en"
    },
    "code_en": {
      "hf_path": "bigcode/the-stack",
      "split": "train",
      "language_filter": "en"
}
}

dataset_config_path = os.path.join(PROJECT_ROOT, "config", "dataset_sources.json")
with open(dataset_config_path, "w") as f:
    json.dump(dataset_sources, f, indent=4)

print("Dataset sources saved to:", dataset_config_path)
print(json.dumps(dataset_sources, indent=4))

Dataset sources saved to: /content/drive/MyDrive/oax_1B/config/dataset_sources.json
{
    "slimpajama_en": {
        "hf_path": "cerebras/SlimPajama-627B",
        "split": "train",
        "language_filter": "en"
    },
    "wikipedia_en": {
        "hf_path": "wikimedia/wikipedia",
        "split": "20231101.en",
        "language_filter": "en"
    },
    "c4_owt_en": {
        "hf_path": "allenai/c4",
        "split": "en",
        "language_filter": "en"
    },
    "stackexchange_en": {
        "hf_path": "bigscience-data/roots_code_stackexchange",
        "split": "train",
        "language_filter": "en"
    },
    "code_en": {
        "hf_path": "bigcode/the-stack",
        "split": "train",
        "language_filter": "en"
    }
}


# **SECTION 4 — TOKENIZER SAMPLING**

In [34]:
!pip install -q huggingface_hub
from huggingface_hub import login

login()


In [47]:
import os
import json
from datasets import load_dataset

PROJECT_ROOT = "/content/drive/MyDrive/oax_1B"
dataset_config_path = os.path.join(PROJECT_ROOT, "config", "dataset_sources.json")

with open(dataset_config_path, "r") as f:
    dataset_sources = json.load(f)

# Sampling amount per dataset
sample_sizes = {
    "slimpajama_en": 50000,
    "wikipedia_en": 10000,
    "c4_owt_en": 10000,
    "stackexchange_en": 10000,
    "code_en": 10000
}

# Output corpus file
os.makedirs(os.path.join(PROJECT_ROOT, "tokenizer"), exist_ok=True)
corpus_path = os.path.join(PROJECT_ROOT, "tokenizer", "tokenizer_corpus.txt")

def is_english(text):
    try:
        text.encode("ascii")
        return True
    except:
        return False


with open(corpus_path, "w", encoding="utf-8") as corpus_file:

    for key, cfg in dataset_sources.items():
        hf_path = cfg["hf_path"]
        split = cfg["split"]
        target_samples = sample_sizes[key]

        print(f"\nSampling {target_samples} examples from {key} ({hf_path})...")

        # SPECIAL CASE HANDLING
        if key == "wikipedia_en":
            ds = load_dataset(hf_path, cfg["split"], split="train", streaming=True)

        elif key == "c4_owt_en":
            ds = load_dataset(hf_path, "en", split="train", streaming=True)

        else:
            ds = load_dataset(hf_path, split=split, streaming=True)

        count = 0

        for item in ds:
            text = (
                item.get("text")
                or item.get("content")
                or item.get("body")
                or item.get("question")
                or item.get("answer")
            )

            if not text:
                continue
            if not is_english(text):
                continue

            corpus_file.write(text.replace("\n", " ") + "\n")
            count += 1

            if count >= target_samples:
                break

        print(f"Collected {count} samples from {key}")

print("\nTokenizer sampling complete!")
print("Corpus saved at:", corpus_path)



Sampling 50000 examples from slimpajama_en (cerebras/SlimPajama-627B)...


Resolving data files:   0%|          | 0/59166 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31428 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31411 [00:00<?, ?it/s]

Collected 50000 samples from slimpajama_en

Sampling 10000 examples from wikipedia_en (wikimedia/wikipedia)...


...

Collected 10000 samples from c4_owt_en

Sampling 10000 examples from stackexchange_en (bigscience-data/roots_code_stackexchange)...


Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Collected 10000 samples from stackexchange_en

Sampling 10000 examples from code_en (bigcode/the-stack)...


Resolving data files:   0%|          | 0/6824 [00:00<?, ?it/s]

Collected 10000 samples from code_en

Tokenizer sampling complete!
Corpus saved at: /content/drive/MyDrive/oax_1B/tokenizer/tokenizer_corpus.txt


# **SECTION 5 — TOKENIZER TRAINING**

In [49]:
import sentencepiece as spm
import os

PROJECT_ROOT = "/content/drive/MyDrive/oax_1B"
corpus_path = os.path.join(PROJECT_ROOT, "tokenizer", "tokenizer_corpus.txt")
sp_model_prefix = os.path.join(PROJECT_ROOT, "tokenizer", "spm_oax_1B")

vocab_size = 50000

train_cmd = (
    f"--input={corpus_path} "
    f"--model_prefix={sp_model_prefix} "
    f"--vocab_size={vocab_size} "
    f"--character_coverage=1.0 "
    f"--model_type=unigram "
    f"--normalization_rule_name=nfkc "
    f"--input_sentence_size=3000000 "
    f"--shuffle_input_sentence=true "
    f"--pad_id=0 --pad_piece=<pad> "
    f"--unk_id=1 --unk_piece=<unk> "
    f"--bos_id=2 --bos_piece=<bos> "
    f"--eos_id=3 --eos_piece=<eos> "
    f"--num_threads=8"
)

print("Training SentencePiece tokenizer...")
spm.SentencePieceTrainer.Train(train_cmd)

print("\n Tokenizer training complete!")
print("Generated files:")
print(f"- {sp_model_prefix}.model")
print(f"- {sp_model_prefix}.vocab")

Training SentencePiece tokenizer...

 Tokenizer training complete!
Generated files:
- /content/drive/MyDrive/oax_1B/tokenizer/spm_oax_1B.model
- /content/drive/MyDrive/oax_1B/tokenizer/spm_oax_1B.vocab


# **SECTION 6 — MODEL CONFIGURATION**

In [51]:
import sentencepiece as spm

tokenizer_path = "/content/drive/MyDrive/oax_1B/tokenizer/spm_oax_1B.model"

sp = spm.SentencePieceProcessor()
sp.load(tokenizer_path)

print("Tokenizer loaded")

# ---- Simple test ----
test_text = "Hello, how are you today?"

ids = sp.encode(test_text)
back = sp.decode(ids)

print("\nInput text:")
print(test_text)

print("\nToken IDs:")
print(ids)

print("\nDecoded text:")
print(back)


Tokenizer loaded

Input text:
Hello, how are you today?

Token IDs:
[6618, 5, 125, 29, 23, 566, 49]

Decoded text:
Hello, how are you today?


# **SECTION 7 — PRETRAIN DATALOADER CONSTRUCTION**

# **SECTION 8 — PRETRAIN TRAINING LOOP (TARGET: 20B TOKENS)**

# **SECTION 9 — PRETRAIN MONITORING & EVALUATION**

# **SECTION 10 — SAVE 1B ENGLISH BASE LM (20B TOKEN VERSION)**

# **SECTION 11 — SFT DATASET PREPARATION**

# **SECTION 12 — SFT FORMAT DEFINITION**

# **SECTION 13 — SFT TRAINING (BASE → ASSISTANT)**

# **SECTION 14 — SFT EVALUATION**


# **SECTION 15 — SAVE FINAL 1B BASE ASSISTANT MODEL**